In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
loader = DataLoader(dataset, batch_size=50000, shuffle=False)

images, _ = next(iter(loader))
mean = images.mean(dim=[0, 2, 3])
std = images.std(dim=[0, 2, 3])
print("Mean:", mean)
print("Std:", std)


100%|██████████| 170M/170M [00:08<00:00, 20.0MB/s]


Mean: tensor([0.4914, 0.4822, 0.4465])
Std: tensor([0.2470, 0.2435, 0.2616])


In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load CIFAR-10
train_val_dataset = datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

# Class names
class_names = train_val_dataset.classes
print(f"✅ Class names: {class_names}")

# Split training into train/val
train_size = int(0.8 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Sanity check
print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Validation batches: {len(val_loader)}")
print(f"✅ Test batches: {len(test_loader)}")

# Peek at shape
for images, labels in train_loader:
    print(f"✅ Image batch shape: {images.shape}")  # Expected: [64, 3, 32, 32]
    print(f"✅ Label batch shape: {labels.shape}")  # Expected: [64]
    break


✅ Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
✅ Train batches: 625
✅ Validation batches: 157
✅ Test batches: 157
✅ Image batch shape: torch.Size([64, 3, 32, 32])
✅ Label batch shape: torch.Size([64])


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class LeNet_CIFAR10(nn.Module):
    def __init__(self):
        super(LeNet_CIFAR10, self).__init__()

        # Convolutional Block
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5),     # (3,32,32) → (6,28,28)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                        # → (6,14,14)

            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5),    # → (16,10,10)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                         # → (16,5,5)
        )

        # Fully Connected Classifier
        self.fc_layers = nn.Sequential(
            nn.Flatten(),                    # → 16*5*5 = 400
            nn.Linear(400, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10)                # 10 output logits
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [4]:
import torch
import torch.optim as optim
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Instantiate model, loss, optimizer
lenet_model = LeNet_CIFAR10().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters(), lr=0.001)

# For tracking metrics
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []


In [ ]:
epochs = 50

for epoch in range(epochs):
    lenet_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = lenet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    # Validation
    lenet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = lenet_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_losses.append(val_loss / val_total)
    val_accuracies.append(val_correct / val_total)

    print(f"Epoch [{epoch+1}/{epochs}] - Train Acc: {epoch_acc:.4f}, Val Acc: {val_correct / val_total:.4f}")

Epoch [1/50] - Train Acc: 0.3953, Val Acc: 0.4734
Epoch [2/50] - Train Acc: 0.5014, Val Acc: 0.5325
Epoch [3/50] - Train Acc: 0.5495, Val Acc: 0.5598
Epoch [4/50] - Train Acc: 0.5859, Val Acc: 0.5847
Epoch [5/50] - Train Acc: 0.6101, Val Acc: 0.5946
Epoch [6/50] - Train Acc: 0.6351, Val Acc: 0.6158
Epoch [7/50] - Train Acc: 0.6516, Val Acc: 0.6163
Epoch [8/50] - Train Acc: 0.6682, Val Acc: 0.6108
Epoch [9/50] - Train Acc: 0.6860, Val Acc: 0.6320
Epoch [10/50] - Train Acc: 0.7006, Val Acc: 0.6295
Epoch [11/50] - Train Acc: 0.7124, Val Acc: 0.6343
Epoch [12/50] - Train Acc: 0.7242, Val Acc: 0.6359
Epoch [13/50] - Train Acc: 0.7318, Val Acc: 0.6376
Epoch [14/50] - Train Acc: 0.7450, Val Acc: 0.6424
Epoch [15/50] - Train Acc: 0.7590, Val Acc: 0.6315
Epoch [16/50] - Train Acc: 0.7660, Val Acc: 0.6345
Epoch [17/50] - Train Acc: 0.7761, Val Acc: 0.6396
Epoch [18/50] - Train Acc: 0.7827, Val Acc: 0.6321
Epoch [19/50] - Train Acc: 0.7933, Val Acc: 0.6323
Epoch [20/50] - Train Acc: 0.8004, Val A

In [ ]:
import torch.nn as nn

class AlexNetMini(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),  # [B, 3, 32, 32] → [B, 64, 32, 32]
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # [B, 64, 32, 32] → [B, 64, 16, 16]

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # [B, 64, 16, 16] → [B, 128, 16, 16]
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # → [B, 128, 8, 8]

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1), # → [B, 256, 8, 8]
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), # → [B, 256, 8, 8]
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), # → [B, 256, 8, 8]
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                            # → [B, 256, 4, 4]
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten: [B, 256, 4, 4] → [B, 4096]
        x = self.classifier(x)
        return x

# Instantiate model
alexnet_model = AlexNetMini().to(device)
print(alexnet_model)


AlexNetMini(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=4096, out_features=1024, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bia

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate model, loss, optimizer
alexnet_model = AlexNetMini().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet_model.parameters(), lr=0.001)

# Track metrics
epochs = 50 # Do not change this because it is just to illustrate how long it takes to train just one epoch without a GPU.
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(epochs):
    # Training Phase
    alexnet_model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = alexnet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Validation Phase
    alexnet_model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = alexnet_model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_losses.append(val_loss / val_total)
    val_accuracies.append(val_correct / val_total)

    print(f"Epoch [{epoch+1}/{epochs}] - "
          f"Train Acc: {train_acc:.4f}, Val Acc: {val_correct / val_total:.4f}")

Epoch [1/10] - Train Acc: 0.3669, Val Acc: 0.5264, Time: 715.12 sec
Epoch [2/10] - Train Acc: 0.5495, Val Acc: 0.5992, Time: 766.96 sec
Epoch [3/10] - Train Acc: 0.6253, Val Acc: 0.6388, Time: 757.68 sec
Epoch [4/10] - Train Acc: 0.6754, Val Acc: 0.6967, Time: 754.23 sec
Epoch [5/10] - Train Acc: 0.7116, Val Acc: 0.7061, Time: 753.35 sec
Epoch [6/10] - Train Acc: 0.7378, Val Acc: 0.7054, Time: 751.01 sec
Epoch [7/10] - Train Acc: 0.7611, Val Acc: 0.7321, Time: 748.22 sec


In [ ]:
# Get CUDA version
!nvcc --version
# Get your library versions
!pip list | grep -E "torch|numpy|scipy"

/bin/bash: line 1: nvcc: command not found
numpy                                    2.0.2
scipy                                    1.16.3
torch                                    2.10.0+cpu
torchao                                  0.10.0
torchaudio                               2.10.0+cpu
torchcodec                               0.10.0
torchdata                                0.11.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cpu


In [ ]:
!nvcc --version
!python -c "import torch; print(torch.version.cuda)"

/bin/bash: line 1: nvcc: command not found
None
